In [2]:
# %% [markdown]
# # Notebook 02 — Benchmark Comparaison & Validation du Seuil 0.85
#
# **Objectif** : Mesurer les performances de détection sur des fichiers
# musicaux réels (GTZAN) et valider que le seuil 0.85 est optimal.
#
# **Corpus** : 
# - `data/references/` : œuvres originales (2)
# - `data/pirates/`    : copies dégradées (2)
# - `data/legal/`      : autres morceaux distincts (2)

# %%
import sys, os, time, gc
import numpy as np
import soundfile as sf
import tempfile
import librosa
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from backend.core.fingerprint_engine import FingerprintEngine
from backend.db.fingerprint_index    import FingerprintIndex
from backend.config                  import settings

# %%
engine = FingerprintEngine()
engine.load_grafprint_model()
print(f"Moteur : {'GraFPrint GNN' if engine.model else 'Fallback Librosa'}")

# %% [markdown]
# ## 1. Chargement du corpus réel (tronqué à 10 s)

# %%
DATA_DIR = Path("../data")

reference_files = sorted((DATA_DIR / "references").glob("*"))
pirate_files    = sorted((DATA_DIR / "pirates").glob("*"))
legal_files     = sorted((DATA_DIR / "legal").glob("*"))

print("Fichiers utilisés :")
for label, files in [("Références", reference_files), ("Pirates", pirate_files), ("Légaux", legal_files)]:
    print(f"\n  {label} ({len(files)} fichiers) :")
    for f in files:
        # Lecture rapide pour obtenir la durée sans charger tout le signal
        y, sr = librosa.load(str(f), sr=None, mono=True, duration=None)
        duration = len(y) / sr
        print(f"    {f.name:35s}  {duration:.1f}s")

Moteur : GraFPrint GNN
Fichiers utilisés :

  Références (10 fichiers) :
    blues.00001.wav                      30.0s
    classical.00001.wav                  30.0s
    country.00001.wav                    30.0s
    disco.00008.wav                      30.2s
    hiphop.00010.wav                     30.0s
    jazz.00001.wav                       30.0s
    metal.00001.wav                      30.0s
    pop.00001.wav                        30.0s
    reggae.00001.wav                     30.0s
    rock.00001.wav                       30.0s

  Pirates (10 fichiers) :
    pirate_blues.00001.wav               30.0s
    pirate_classical.00001.wav           30.0s
    pirate_country.00001.wav             30.0s
    pirate_disco.00008.wav               30.2s
    pirate_hiphop.00010.wav              30.0s
    pirate_jazz.00001.wav                30.0s
    pirate_metal.00001.wav               30.0s
    pirate_pop.00001.wav                 30.0s
    pirate_reggae.00001.wav              30.0s
    pir

In [3]:
# %% [markdown]
# ## 2. Construction de l'index de référence

# %%
index_path = tempfile.mktemp(suffix=".json")
index = FingerprintIndex(Path(index_path))

print("Indexation des œuvres originales...")
indexed_hashes = []
for i, path in enumerate(reference_files):
    emb = engine.extract_fingerprint(path)
    h = index.add(
        embedding  = emb,
        title      = f"Reference_{i+1:02d}",
        artist     = f"Artiste_Ref_{i+1:02d}",
        ipfs_cid   = f"QmRef{i:04d}",
        tx_hash    = f"0xREF{i:04d}",
        recipients = ["0xABC123"],
        shares     = [100]
    )
    indexed_hashes.append(h)
    print(f"  [{i+1:2d}/{len(reference_files)}] {h[:16]}... ({Path(path).name})")
    del emb
    gc.collect()

print(f"\nIndex : {index.count()} empreintes")

Indexation des œuvres originales...
  [ 1/10] 256fe2ce52e2ab60... (blues.00001.wav)
  [ 2/10] 9625342a233b07bd... (classical.00001.wav)
  [ 3/10] acd0830d558b4a3c... (country.00001.wav)
  [ 4/10] 3aece2568c87b4de... (disco.00008.wav)
  [ 5/10] 6bcc2c5a519e0020... (hiphop.00010.wav)
  [ 6/10] 14ad6ecdb857334f... (jazz.00001.wav)
  [ 7/10] 2d56712e1b596c5c... (metal.00001.wav)
  [ 8/10] c5b44b4ede73708b... (pop.00001.wav)
  [ 9/10] 8ae8ecce066f2a75... (reggae.00001.wav)
  [10/10] 20b4b6d2e6850aaf... (rock.00001.wav)

Index : 10 empreintes


In [4]:
# %% [markdown]
# ## 3. Évaluation sur le corpus réel (seuil 0.85)

# %%
def evaluate(query_files, ground_truth_positive, threshold):
    tp = tn = fp = fn = 0
    scores = []
    for path in query_files:
        emb = engine.extract_fingerprint(path)
        results = index.search(emb, top_k=1)
        score = results[0]["score"] if results else 0.0
        scores.append(score)
        predicted = score >= threshold

        if ground_truth_positive and predicted:
            tp += 1
        elif ground_truth_positive and not predicted:
            fn += 1
        elif not ground_truth_positive and predicted:
            fp += 1
        else:
            tn += 1
        del emb
        gc.collect()
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "scores": scores}

THRESHOLD = 0.85
res_pirates = evaluate(pirate_files, True, THRESHOLD)
res_legaux  = evaluate(legal_files, False, THRESHOLD)

tp = res_pirates["tp"]; fn = res_pirates["fn"]
fp = res_legaux["fp"]; tn = res_legaux["tn"]

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

print(f"\nSeuil : {THRESHOLD}")
print(f"  TP : {tp}, TN : {tn}, FP : {fp}, FN : {fn}")
print(f"  Précision : {precision:.4f}, Rappel : {recall:.4f}, F1 : {f1:.4f}")

# Afficher les scores individuels
print("\nScores pirates (attendu ≥ seuil) :")
for p, s in zip(pirate_files, res_pirates["scores"]):
    print(f"  {Path(p).name:30s} : {s:.4f}")
print("\nScores légaux (attendu < seuil) :")
for p, s in zip(legal_files, res_legaux["scores"]):
    print(f"  {Path(p).name:30s} : {s:.4f}")


Seuil : 0.85
  TP : 10, TN : 10, FP : 0, FN : 0
  Précision : 1.0000, Rappel : 1.0000, F1 : 1.0000

Scores pirates (attendu ≥ seuil) :
  pirate_blues.00001.wav         : 0.9282
  pirate_classical.00001.wav     : 0.9304
  pirate_country.00001.wav       : 0.8541
  pirate_disco.00008.wav         : 0.9449
  pirate_hiphop.00010.wav        : 0.9777
  pirate_jazz.00001.wav          : 0.9232
  pirate_metal.00001.wav         : 0.9745
  pirate_pop.00001.wav           : 0.9405
  pirate_reggae.00001.wav        : 0.9425
  pirate_rock.00001.wav          : 0.9654

Scores légaux (attendu < seuil) :
  blues.00005.wav                : 0.6871
  classical.00007.wav            : 0.6887
  country.00006.wav              : 0.7666
  disco.00012.wav                : 0.7492
  hiphop.00001.wav               : 0.7029
  jazz.00010.wav                 : 0.6949
  metal.00011.wav                : 0.8485
  pop.00005.wav                  : 0.7771
  reggae.00012.wav               : 0.7420
  rock.00013.wav               

In [5]:
# %% [markdown]
# ## 4. Courbe précision-rappel (optimisée)

# %%
# Pré-calcul des scores (une seule extraction)
pirate_scores = res_pirates["scores"]
legal_scores  = res_legaux["scores"]

thresholds = np.arange(0.50, 1.01, 0.05)
results_by_threshold = []

for t in thresholds:
    tp = sum(1 for s in pirate_scores if s >= t)
    fn = len(pirate_scores) - tp
    fp = sum(1 for s in legal_scores if s >= t)
    tn = len(legal_scores) - fp

    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
    results_by_threshold.append({"threshold": t, "precision": p, "recall": r, "f1": f1,
                                 "tp": tp, "fp": fp, "fn": fn, "tn": tn})

print(f"{'Seuil':8s} {'Précision':12s} {'Rappel':10s} {'F1':8s}")
for r in results_by_threshold:
    marker = " ◄ OPTIMAL" if r["f1"] == max(x["f1"] for x in results_by_threshold) else ""
    print(f"  {r['threshold']:.2f}   {r['precision']:10.4f}   {r['recall']:8.4f}   {r['f1']:6.4f}{marker}")
best = max(results_by_threshold, key=lambda x: x["f1"])
print(f"\n→ Meilleur seuil : {best['threshold']:.2f} (F1 = {best['f1']:.4f})")

Seuil    Précision    Rappel     F1      
  0.50       0.5000     1.0000   0.6667
  0.55       0.5000     1.0000   0.6667
  0.60       0.5000     1.0000   0.6667
  0.65       0.5000     1.0000   0.6667
  0.70       0.6250     1.0000   0.7692
  0.75       0.7692     1.0000   0.8696
  0.80       0.9091     1.0000   0.9524
  0.85       1.0000     1.0000   1.0000 ◄ OPTIMAL
  0.90       1.0000     0.9000   0.9474
  0.95       1.0000     0.3000   0.4615
  1.00       0.0000     0.0000   0.0000

→ Meilleur seuil : 0.85 (F1 = 1.0000)


In [6]:
# %% [markdown]
# ## 5. Distribution des scores

# %%
print(f"\nPirates  : mean={np.mean(pirate_scores):.4f}, min={np.min(pirate_scores):.4f}, max={np.max(pirate_scores):.4f}")
print(f"Légaux   : mean={np.mean(legal_scores):.4f}, min={np.min(legal_scores):.4f}, max={np.max(legal_scores):.4f}")
gap = np.min(pirate_scores) - np.max(legal_scores)
print(f"Séparabilité (min_pirate - max_legal) : {gap:.4f} {'✅' if gap > 0 else '⚠️ '}")



Pirates  : mean=0.9381, min=0.8541, max=0.9777
Légaux   : mean=0.7356, min=0.6871, max=0.8485
Séparabilité (min_pirate - max_legal) : 0.0056 ✅


In [7]:
# %% [markdown]
# ## 6. Benchmark de latence (sur 10 itérations)

# %%
test_path = reference_files[0]
test_emb  = engine.extract_fingerprint(test_path)

extract_times = []
for _ in range(10):
    t0 = time.perf_counter()
    engine.extract_fingerprint(test_path)
    extract_times.append(time.perf_counter() - t0)

search_times = []
for _ in range(20):
    t0 = time.perf_counter()
    index.search(test_emb, top_k=1)
    search_times.append(time.perf_counter() - t0)

print(f"Extraction : mean={np.mean(extract_times)*1000:.1f} ms, p95={np.percentile(extract_times,95)*1000:.1f} ms")
print(f"Recherche  : mean={np.mean(search_times)*1000:.3f} ms")

Extraction : mean=14729.9 ms, p95=15266.3 ms
Recherche  : mean=0.041 ms


In [8]:
# %% [markdown]
# ## 7. Résumé

# %%
print("="*60)
print(f"Corpus : {len(reference_files)} refs, {len(pirate_files)} pirates, {len(legal_files)} légaux")
print(f"Seuil 0.85 → P={precision:.4f}, R={recall:.4f}, F1={f1:.4f}")
print(f"Seuil optimal : {best['threshold']:.2f} (F1={best['f1']:.4f})")
print("="*60)

# Nettoyage
# for lst in [reference_files, pirate_files, legal_files]:
#     for p in lst:
#         try: os.unlink(p)
#         except: pass
# os.unlink(index_path)

Corpus : 10 refs, 10 pirates, 10 légaux
Seuil 0.85 → P=1.0000, R=1.0000, F1=0.0000
Seuil optimal : 0.85 (F1=1.0000)
